# Kani Big v2 — free GPU training
Continue **your own Kani weights**, expanding 12 layers to 24 (~173M parameters) and context to 2,048 tokens. New residual blocks initially preserve the original predictions. This is an experimental capacity upgrade, **not evidence of GPT-4-level ability**.

Use **Runtime → Change runtime type → T4 GPU** on Colab's free tier. Do not purchase compute units. Availability and session limits vary. This notebook uses MLX for arrays/autograd; no external pretrained model or tokenizer is downloaded.

**Inputs:** `kani-cloud.zip`, prepared from the local repository, its own checkpoint, and the existing ~164M-token corpus. Repeated training samples are not new data. **Outputs:** resumable `.npz` checkpoints and progress under your Google Drive `KaniTraining/kani-big-v2` folder.

Run the cells in order. Google Drive access is needed to retain checkpoints when a runtime ends. The default session budget is three hours; rerun in a later free session to continue the same 20,000-update experiment.

## 1. Check the GPU and install the training library

In [ ]:
import csv, json, os, platform, subprocess, sys
from pathlib import Path
raw = subprocess.check_output(["nvidia-smi", "--query-gpu=name,compute_cap,driver_version", "--format=csv,noheader"], text=True)
gpus = list(csv.reader(raw.splitlines()))
name, capability, driver = [item.strip() for item in gpus[0]]
if float(capability) < 7.5:
    raise RuntimeError(f"{name} is unsupported by this MLX CUDA build. Select a T4 runtime (P100 is unsupported).")
if tuple(int(x) for x in driver.split(".")) < (550, 54, 14):
    raise RuntimeError("This runtime's NVIDIA driver is too old for MLX CUDA 12.")
print({"GPU": name, "compute_capability": capability, "driver": driver})
subprocess.check_call([sys.executable, "-m", "pip", "install", "mlx[cuda12]==0.32.4", "numpy>=1.24,<3"])
import mlx.core as mx
if not mx.cuda.is_available():
    raise RuntimeError("MLX cannot access CUDA. Stop here; do not silently train on CPU.")
mx.set_default_device(mx.gpu)
print("MLX", mx.__version__, "device", mx.default_device())

## 2. Connect persistent storage
Colab will request Google Drive access. Review and approve it yourself. Only this notebook’s `KaniTraining` folder is used by the code below.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
STORE = Path("/content/drive/MyDrive/KaniTraining")
STORE.mkdir(parents=True, exist_ok=True)
BACKUP = STORE / "kani-big-v2"
BUNDLE = STORE / "kani-cloud.zip"
print("Checkpoint backup:", BACKUP)

## 3. Upload the prepared bundle (first session only)
Choose the local `training-runs/cloud/kani-cloud.zip`. Future sessions reuse its Drive copy.

In [ ]:
import shutil
if not BUNDLE.exists():
    from google.colab import files
    uploaded = files.upload()
    names = list(uploaded)
    del uploaded
    if len(names) != 1 or not names[0].endswith(".zip"):
        raise ValueError("Upload exactly the prepared kani-cloud.zip file.")
    temporary = BUNDLE.with_suffix(".uploading")
    shutil.copyfile(names[0], temporary)
    temporary.replace(BUNDLE)
print("Bundle:", BUNDLE, "bytes:", BUNDLE.stat().st_size)

## 4. Verify every file and unpack
The bundle contains a specific source checkpoint and records its training step. It does not depend on changes in the local Mac’s running training job.

In [ ]:
import hashlib, zipfile
ROOT = Path("/content/kani-cloud")
ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE) as archive:
    metadata = json.loads(archive.read("bundle.json"))
    if metadata.get("format") != "kani-cloud-bundle-v1":
        raise ValueError("Unexpected bundle format")
    if set(archive.namelist()) != set(metadata["included"]) | {"bundle.json"}:
        raise ValueError("Unexpected files in bundle")
    for relative, expected in metadata["included"].items():
        target = (ROOT / relative).resolve()
        if not target.is_relative_to(ROOT.resolve()):
            raise ValueError("Unsafe archive path")
        target.parent.mkdir(parents=True, exist_ok=True)
        digest = hashlib.sha256()
        size = 0
        with archive.open(relative) as source, target.open("wb") as output:
            while chunk := source.read(1 << 20):
                digest.update(chunk)
                size += len(chunk)
                output.write(chunk)
        if digest.hexdigest() != expected["sha256"] or size != expected["bytes"]:
            raise ValueError("Bundle checksum mismatch: " + relative)
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Verified", len(metadata["included"]), "files; source checkpoint step", metadata["source_step"])
print("Original model parameters:", metadata["source_parameters"])

## 5. Grow Kani or restore the existing run
The old checkpoint stays untouched. New blocks initially behave as identity transformations. Restoring uses the latest saved optimizer and random state.

In [ ]:
RUN = ROOT / "training-runs/kani-big-v2"
RUN.mkdir(parents=True, exist_ok=True)
if not (RUN / "latest.npz").exists() and (BACKUP / "latest.npz").exists():
    for filename in ("latest.npz", "best.npz", "status.json", "metrics.jsonl"):
        source = BACKUP / filename
        if source.exists():
            shutil.copy2(source, RUN / filename)
GROWN = ROOT / "checkpoints/grown.npz"
if not (RUN / "latest.npz").exists() and not GROWN.exists():
    subprocess.check_call([sys.executable, "-m", "kelus.grow", "--source", "checkpoints/parent.npz",
                           "--out", str(GROWN), "--layers", "24", "--context", "2048"])
print("Resume" if (RUN / "latest.npz").exists() else "New continuation", RUN)

## 6. Train with a live progress bar
Target: **20,000 optimizer updates** across sessions. Each update uses four microbatches of one 2,048-token sequence (8,192 training tokens/update). This is ~164M additional token presentations, sampled from the existing corpus. Keep the recipe unchanged when resuming. The session budget can change.

Checkpoints save every 250 updates and are copied to Drive. A sudden runtime termination can lose work since the last backup. Interrupting this cell sends SIGINT to request a checkpoint; allow it to finish saving before disconnecting.

In [ ]:
import signal, time
from IPython.display import display
import ipywidgets as widgets
TOTAL_STEPS = 20000
SESSION_SECONDS = 3 * 60 * 60
command = [sys.executable, "-u", "-m", "kelus.gpu_train", "pretrain", "--data", "data/big-v1",
           "--run", str(RUN), "--steps", str(TOTAL_STEPS), "--batch", "1", "--accumulate", "4",
           "--lr", "0.0001", "--seed", "2041", "--checkpoint-every", "250", "--validation-batches", "8",
           "--gradient-checkpointing", "--max-seconds", str(SESSION_SECONDS), "--memory-gb", "12",
           "--backup-dir", str(BACKUP)]
command += ["--resume"] if (RUN / "latest.npz").exists() else ["--init-from", str(GROWN)]
progress = widgets.IntProgress(value=0, min=0, max=TOTAL_STEPS, description="Training:")
label = widgets.HTML("Starting…")
display(progress, label)
log_path = RUN.with_suffix(".log")
with log_path.open("a") as log:
    process = subprocess.Popen(command, cwd=ROOT, stdout=log, stderr=subprocess.STDOUT)
    try:
        while process.poll() is None:
            status_path = RUN / "status.json"
            if status_path.exists():
                status = json.loads(status_path.read_text())
                progress.value = status["step"]
                eta = status.get("eta_seconds")
                remaining = f" · estimated {eta / 3600:.1f} hours to full target" if eta else ""
                label.value = f'{status["step"]:,} / {TOTAL_STEPS:,} · {status["state"]}{remaining}'
            time.sleep(5)
    except KeyboardInterrupt:
        process.send_signal(signal.SIGINT)
        print("Saving checkpoint; waiting for the trainer to exit…")
        process.wait()
if process.returncode:
    print("\n".join(log_path.read_text(errors="replace").splitlines()[-25:]))
    raise RuntimeError(f"Trainer exited {process.returncode}; previous Drive backups are retained")
status = json.loads((RUN / "status.json").read_text())
progress.value = status["step"]
label.value = f'{status["state"]}: {status["step"]:,} / {TOTAL_STEPS:,}; backup: {BACKUP}'
print(json.dumps(status, indent=2))

## 7. Inspect progress and keep the checkpoint
“Stopped” means the session budget ended; rerun in another available free session to continue. “Complete” means the update budget finished, not that the model passed capability tests. These are raw-text weights; instruction tuning and independent coding/reasoning evaluations are still needed. Do not automatically replace the released Kani/Cide models.

In [ ]:
latest = BACKUP / "latest.npz"
if latest.exists():
    print("Persistent checkpoint:", latest, "bytes:", latest.stat().st_size)
else:
    raise RuntimeError("No persistent checkpoint found. Inspect training output before ending the runtime.")
print("Training log:", log_path)
print("After saving, use Runtime → Disconnect and delete runtime to release the free GPU.")

## References and validation
- [Colab free resources and dynamic limits](https://research.google.com/colaboratory/faq.html)
- [MLX CUDA installation and supported GPUs](https://ml-explore.github.io/mlx/build/html/install.html#cuda)
- Source/data provenance and file hashes: `bundle.json` in the uploaded archive.
- Local tests verify growth preserves initial predictions, new layers receive gradients, checkpointed gradients match, and interrupted training resumes weights/optimizer/RNG consistently.

**Execution status:** notebook structure and code cells can be validated locally. Actual CUDA execution requires a Colab T4 runtime; no remote benchmark result is claimed by this notebook template.